# L12: 自定义工具开发实战

> 从零开始开发功能完整的 Agent 工具

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
import httpx
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Any

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.tools import Tool, register_tool, ToolRegistry
print("✓ 模块导入成功")

## 12.1 工具开发流程

### 从需求到工具的完整路径

```
1. 需求分析 → 确定工具功能
   ↓
2. 设计参数 → 定义输入输出
   ↓
3. 实现逻辑 → 编写 execute 方法
   ↓
4. 测试验证 → 确保功能正确
   ↓
5. 注册使用 → 添加到工具库
```

## 12.2 实战 1：天气查询工具

### 需求
- 查询指定城市的天气
- 支持当前天气和预报
- 支持摄氏度/华氏度切换

In [ ]:
@register_tool
class WeatherTool(Tool):
    """天气查询工具（模拟实现）"""
    
    name = "get_weather"
    description = "查询指定城市的天气信息，包括温度、天气状况、湿度等"
    
    parameters = {
        "city": {
            "type": "string",
            "description": "城市名称（中文或英文），如 '北京' 或 'Beijing'",
            "required": True
        },
        "unit": {
            "type": "string",
            "description": "温度单位",
            "required": False,
            "default": "celsius",
            "enum": ["celsius", "fahrenheit"]
        }
    }
    
    # 模拟天气数据（实际开发中应调用真实 API）
    _mock_data = {
        "北京": {"temp": 22, "condition": "晴", "humidity": 45},
        "上海": {"temp": 25, "condition": "多云", "humidity": 60},
        "深圳": {"temp": 28, "condition": "阵雨", "humidity": 75},
        "Beijing": {"temp": 22, "condition": "Sunny", "humidity": 45},
        "Shanghai": {"temp": 25, "condition": "Cloudy", "humidity": 60},
    }
    
    async def execute(self, city: str, unit: str = "celsius") -> str:
        """
        查询天气
        
        Args:
            city: 城市名称
            unit: 温度单位 (celsius/fahrenheit)
            
        Returns:
            天气信息字符串
        """
        # 获取天气数据
        data = self._mock_data.get(city)
        
        if not data:
            return f"抱歉，没有找到城市 '{city}' 的天气信息"
        
        # 转换温度单位
        temp = data["temp"]
        if unit == "fahrenheit":
            temp = temp * 9/5 + 32
            temp_symbol = "°F"
        else:
            temp_symbol = "°C"
        
        # 格式化输出
        result = (
            f"🌍 {city}\n"
            f"🌡️ 温度: {temp}{temp_symbol}\n"
            f"☁️ 天气: {data['condition']}\n"
            f"💧 湿度: {data['humidity']}%"
        )
        
        return result

# 测试天气工具
async def test_weather():
    tool = ToolRegistry.get("get_weather")
    
    print("=== 天气工具测试 ===\n")
    
    cities = ["北京", "上海", "深圳", "Unknown"]
    for city in cities:
        result = await tool.execute(city=city)
        print(f"{result}\n")

await test_weather()

## 12.3 实战 2：API 请求工具

### 需求
- 发送 HTTP GET/POST 请求
- 支持自定义请求头
- 返回响应内容

In [ ]:
@register_tool
class HttpRequestTool(Tool):
    """HTTP 请求工具"""
    
    name = "http_request"
    description = "发送 HTTP 请求获取数据，支持 GET 和 POST 方法"
    
    parameters = {
        "url": {
            "type": "string",
            "description": "请求的完整 URL",
            "required": True
        },
        "method": {
            "type": "string",
            "description": "HTTP 请求方法",
            "required": False,
            "default": "GET",
            "enum": ["GET", "POST", "PUT", "DELETE"]
        },
        "headers": {
            "type": "object",
            "description": "请求头（JSON 格式）",
            "required": False
        },
        "body": {
            "type": "string",
            "description": "请求体（JSON 字符串，仅 POST/PUT 有效）",
            "required": False
        }
    }
    
    async def execute(
        self, 
        url: str, 
        method: str = "GET",
        headers: Optional[Dict[str, str]] = None,
        body: Optional[str] = None
    ) -> str:
        """
        发送 HTTP 请求
        """
        try:
            async with httpx.AsyncClient(timeout=30.0) as client:
                # 准备请求参数
                request_headers = headers or {}
                
                if method in ["POST", "PUT"] and body:
                    request_headers["Content-Type"] = "application/json"
                    response = await client.request(
                        method=method,
                        url=url,
                        headers=request_headers,
                        content=body
                    )
                else:
                    response = await client.request(
                        method=method,
                        url=url,
                        headers=request_headers
                    )
                
                # 返回结果
                return json.dumps({
                    "status_code": response.status_code,
                    "headers": dict(response.headers),
                    "body": response.text[:1000]  # 限制返回长度
                }, ensure_ascii=False, indent=2)
                
        except httpx.TimeoutError:
            return "错误: 请求超时"
        except httpx.HTTPError as e:
            return f"错误: HTTP 请求失败 - {str(e)}"
        except Exception as e:
            return f"错误: {str(e)}"

# 测试 HTTP 工具
async def test_http():
    tool = ToolRegistry.get("http_request")
    
    print("=== HTTP 工具测试 ===\n")
    
    # 测试 GET 请求
    result = await tool.execute(
        url="https://httpbin.org/get",
        method="GET"
    )
    print("GET 请求结果:")
    print(result[:300] + "...\n")

# await test_http()  # 取消注释以测试

## 12.4 实战 3：数据处理工具

### 需求
- 对数据进行统计
- 支持多种统计类型
- 处理 JSON/列表数据

In [ ]:
@register_tool
class DataAnalysisTool(Tool):
    """数据分析工具"""
    
    name = "analyze_data"
    description = "对数值数据进行分析统计，支持求和、平均值、最大值、最小值等"
    
    parameters = {
        "data": {
            "type": "array",
            "description": "数值数组，如 [1, 2, 3, 4, 5]",
            "required": True
        },
        "operation": {
            "type": "string",
            "description": "统计操作类型",
            "required": False,
            "default": "all",
            "enum": ["sum", "avg", "max", "min", "count", "all"]
        }
    }
    
    async def execute(self, data: List[float], operation: str = "all") -> str:
        """
        执行数据分析
        """
        try:
            if not data:
                return "错误: 数据不能为空"
            
            # 验证数据类型
            numbers = []
            for item in data:
                if isinstance(item, (int, float)):
                    numbers.append(float(item))
                elif isinstance(item, str) and item.replace('.', '', 1).replace('-', '', 1).isdigit():
                    numbers.append(float(item))
                else:
                    return f"错误: 数据包含非数值项: {item}"
            
            if not numbers:
                return "错误: 没有有效的数值数据"
            
            # 执行统计
            if operation == "sum":
                result = {"sum": sum(numbers)}
            elif operation == "avg":
                result = {"average": sum(numbers) / len(numbers)}
            elif operation == "max":
                result = {"maximum": max(numbers)}
            elif operation == "min":
                result = {"minimum": min(numbers)}
            elif operation == "count":
                result = {"count": len(numbers)}
            else:  # all
                result = {
                    "count": len(numbers),
                    "sum": sum(numbers),
                    "average": sum(numbers) / len(numbers),
                    "maximum": max(numbers),
                    "minimum": min(numbers)
                }
            
            # 格式化输出
            return json.dumps(result, ensure_ascii=False, indent=2)
            
        except Exception as e:
            return f"错误: 分析失败 - {str(e)}"

# 测试数据分析工具
async def test_analysis():
    tool = ToolRegistry.get("analyze_data")
    
    print("=== 数据分析工具测试 ===\n")
    
    test_data = [23, 45, 67, 89, 12, 34, 56, 78, 90, 1]
    
    print(f"数据: {test_data}\n")
    
    for op in ["sum", "avg", "max", "min", "all"]:
        result = await tool.execute(data=test_data, operation=op)
        print(f"{op.upper()}: {result}\n")

await test_analysis()

## 12.5 实战 4：时间转换工具

### 需求
- 时间戳转日期
- 日期转时间戳
- 时区转换

In [ ]:
@register_tool
class TimeConverterTool(Tool):
    """时间转换工具"""
    
    name = "convert_time"
    description = "时间格式转换工具，支持时间戳与日期字符串互转"
    
    parameters = {
        "value": {
            "type": "string",
            "description": "要转换的时间值（时间戳或日期字符串）",
            "required": True
        },
        "from_type": {
            "type": "string",
            "description": "输入类型",
            "required": False,
            "default": "auto",
            "enum": ["auto", "timestamp", "datetime"]
        },
        "format": {
            "type": "string",
            "description": "输出格式 (full/date/time/iso)",
            "required": False,
            "default": "full"
        }
    }
    
    async def execute(self, value: str, from_type: str = "auto", format: str = "full") -> str:
        """
        执行时间转换
        """
        try:
            dt = None
            
            # 判断输入类型
            if from_type == "auto":
                # 自动检测：纯数字可能是时间戳
                if value.isdigit():
                    from_type = "timestamp"
                else:
                    from_type = "datetime"
            
            # 解析输入
            if from_type == "timestamp":
                timestamp = int(value)
                # 处理秒级和毫秒级时间戳
                if timestamp > 1e12:  # 毫秒
                    timestamp = timestamp / 1000
                dt = datetime.fromtimestamp(timestamp)
            else:
                # 尝试解析日期字符串
                dt = datetime.fromisoformat(value.replace('T', ' ').replace('Z', ''))
            
            # 格式化输出
            if format == "date":
                return dt.strftime("%Y-%m-%d")
            elif format == "time":
                return dt.strftime("%H:%M:%S")
            elif format == "iso":
                return dt.isoformat()
            else:
                return dt.strftime("%Y-%m-%d %H:%M:%S")
                
        except ValueError as e:
            return f"错误: 无法解析日期 - {str(e)}"
        except Exception as e:
            return f"错误: 转换失败 - {str(e)}"

# 测试时间转换工具
async def test_time_converter():
    tool = ToolRegistry.get("convert_time")
    
    print("=== 时间转换工具测试 ===\n")
    
    test_cases = [
        ("1704067200", "auto", "full"),      # Unix 时间戳
        ("1704067200000", "auto", "full"),    # 毫秒时间戳
        ("2024-01-01 00:00:00", "auto", "iso"),  # 日期字符串
        ("1704067200", "timestamp", "date"), # 显式指定类型
    ]
    
    for value, from_type, fmt in test_cases:
        result = await tool.execute(value=value, from_type=from_type, format=fmt)
        print(f"输入: {value} ({from_type}) → {result}")

await test_time_converter()

## 12.6 工具开发最佳实践

### 1. 错误处理

```python
async def execute(self, **kwargs):
    try:
        # 业务逻辑
        result = do_something(**kwargs)
        return str(result)
    except ValidationError as e:
        return f"参数错误: {e}"
    except APIError as e:
        return f"API 调用失败: {e}"
    except Exception as e:
        return f"未知错误: {e}"
```

### 2. 参数验证

```python
async def execute(self, url: str):
    # 验证 URL 格式
    if not url.startswith(('http://', 'https://')):
        return "错误: URL 必须以 http:// 或 https:// 开头"
    
    # 验证长度
    if len(url) > 2000:
        return "错误: URL 过长"
    ...
```

### 3. 超时控制

```python
async def execute(self, query: str):
    try:
        async with asyncio.timeout(10):  # 10秒超时
            result = await slow_api_call(query)
            return result
    except asyncio.TimeoutError:
        return "错误: 请求超时"
```

### 4. 日志记录

```python
import logging

logger = logging.getLogger(__name__)

async def execute(self, **kwargs):
    logger.info(f"调用工具 {self.name}, 参数: {kwargs}")
    try:
        result = await do_work(**kwargs)
        logger.info(f"工具 {self.name} 执行成功")
        return result
    except Exception as e:
        logger.error(f"工具 {self.name} 执行失败: {e}")
        raise
```

### 5. 返回值规范

- **统一返回字符串**：便于 LLM 处理
- **结构化输出**：使用 JSON 格式返回复杂数据
- **错误信息清晰**：说明问题原因，给出解决建议

## 12.7 工具组合模式

### 复合工具：调用其他工具

In [ ]:
@register_tool
class CityInfoTool(Tool):
    """
    城市信息工具 - 复合工具示例
    调用多个子工具获取完整信息
    """
    
    name = "get_city_info"
    description = "获取城市的综合信息，包括天气、时间等"
    
    parameters = {
        "city": {
            "type": "string",
            "description": "城市名称",
            "required": True
        }
    }
    
    async def execute(self, city: str) -> str:
        """
        获取城市综合信息
        """
        results = []
        
        # 调用天气工具
        weather_tool = ToolRegistry.get("get_weather")
        if weather_tool:
            weather = await weather_tool.execute(city=city)
            results.append(f"=== 天气 ===\n{weather}")
        
        # 调用时间工具
        time_tool = ToolRegistry.get("get_current_time")
        if time_tool:
            current_time = await time_tool.execute()
            results.append(f"\n=== 当前时间 ===\n{current_time}")
        
        return "\n".join(results)

# 测试复合工具
async def test_composite():
    tool = ToolRegistry.get("get_city_info")
    
    print("=== 复合工具测试 ===\n")
    
    result = await tool.execute(city="北京")
    print(result)

await test_composite()

## 12.8 常见面试问题

**Q: 工具开发的常见错误有哪些？**
- A:
  1. 没有处理异步（忘记 async/await）
  2. 返回非字符串类型（LLM 无法解析）
  3. 缺少参数验证
  4. 错误处理不完善
  5. 超时控制缺失
  6. 日志记录不足

**Q: 如何调试工具？**
- A:
  1. 编写单元测试
  2. 添加详细日志
  3. 使用 try-except 捕获异常
  4. 打印中间结果
  5. 在 notebook 中单独测试

**Q: 工具可以调用 LLM 吗？**
- A:
  - 可以！这是「Agent 使用 Agent」模式
  - 工具内部可以调用 LLM 进行子任务
  - 注意避免无限循环
  - 要传递 LLM 实例或工厂

**Q: 如何优化工具性能？**
- A:
  1. 添加缓存（相同参数直接返回缓存结果）
  2. 使用连接池（HTTP 请求）
  3. 批量处理（合并多个请求）
  4. 异步并发（多个工具并行执行）
  5. 设置合理的超时时间

**Q: 如何设计可复用的工具？**
- A:
  1. 参数设计灵活（支持多种输入格式）
  2. 功能单一（一个工具只做一件事）
  3. 接口清晰（输入输出明确）
  4. 错误处理完善
  5. 提供详细文档

---

## 总结

本课程学习了自定义工具开发的实战知识：

| 知识点 | 说明 |
|--------|------|
| **开发流程** | 需求分析 → 参数设计 → 实现逻辑 → 测试验证 |
| **错误处理** | try-except、清晰的错误信息 |
| **参数验证** | 类型、格式、长度检查 |
| **超时控制** | asyncio.timeout 防止无限等待 |
| **日志记录** | 便于调试和监控 |
| **复合工具** | 一个工具调用多个子工具 |
| **最佳实践** | 异步、字符串返回、结构化输出 |

**下一步**: L13 将进入记忆系统学习！